# DSE Data Setup & Preprocessing

This notebook loads the Mendeley DSE dataset, cleans it, maps each stock to its DSE sector, and saves processed files that all subsequent notebooks will use.

**Data source:** `Full Raw Data/Adjusted.csv` — one combined file containing all 486 tickers (482 company stocks + 4 market indices), adjusted for dividends and splits.

**Output:** `data/processed/prices.parquet`, `returns.parquet`, `sector_returns.parquet`, `volume.parquet`, `sector_map.csv`

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_FILE  = '../data/raw/Dhaka Stock Exchange End-of-Day Financial Dataset/Full Raw Data/Adjusted.csv'
PROCESSED  = '../data/processed/'

os.makedirs(PROCESSED, exist_ok=True)

## Step 1: Load the Single Combined File

The dataset is already packaged as one CSV with columns:
`Date | Ticker | Open | High | Low | Close | Volume`

Tickers starting with `00` are market indices (DSEX, DS30, etc.).
Everything else is an individual listed company.

In [2]:
raw = pd.read_csv(DATA_FILE, parse_dates=['Date'])
raw.columns = [c.strip() for c in raw.columns]

print(f"Total rows loaded  : {len(raw):,}")
print(f"Unique tickers     : {raw['Ticker'].nunique()}")
print(f"Date range         : {raw['Date'].min().date()}  →  {raw['Date'].max().date()}")
print(f"Columns            : {raw.columns.tolist()}")
print()
print("Sample rows:")
display(raw.sample(5, random_state=1).sort_values('Date'))

Total rows loaded  : 1,080,313
Unique tickers     : 486
Date range         : 2012-10-01  →  2026-01-22
Columns            : ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']

Sample rows:


,Date,Ticker,Open,High,Low,Close,Volume
129412,2014-10-15,PREMIERBAN,5.855452,6.070301,5.694246,5.855452,2072700
356858,2017-10-12,EMERALDOIL,19.900000,19.900000,19.400000,19.600000,267138
432513,2018-09-18,PF1STMF,5.500000,5.500000,5.500000,5.500000,5100
450422,2018-12-03,UTTARABANK,11.322476,11.403641,11.241311,11.322476,179668
819234,2023-04-30,ARGONDENIM,18.200000,18.200000,18.200000,18.200000,82


## Step 2: Separate Market Indices from Company Stocks

The four index tickers (`00DSEX`, `00DS30`, `00DSES`, `00DSMEX`) are market benchmarks — we keep them separate and will not include them in sector analysis.

`00DSEX` is the main broad-market index — used as the benchmark throughout all subsequent notebooks.

In [3]:
# split into indices and company stocks
idx_mask  = raw['Ticker'].str.startswith('00')
indices   = raw[idx_mask].copy()
companies = raw[~idx_mask].copy()

print(f"Index tickers  : {sorted(indices['Ticker'].unique())}")
print(f"Company tickers: {companies['Ticker'].nunique()}")

# DSEX specifically — our market benchmark
dsex = (indices[indices['Ticker'] == '00DSEX']
        [['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
        .sort_values('Date')
        .set_index('Date'))

print(f"\nDSEX date range: {dsex.index.min().date()}  →  {dsex.index.max().date()}")
print(f"DSEX index range: {dsex['Close'].min():.0f}  →  {dsex['Close'].max():.0f}")
print()
display(dsex.head(3))

Index tickers  : ['00DS30', '00DSES', '00DSEX', '00DSMEX']
Company tickers: 482

DSEX date range: 2012-10-01  →  2026-01-22
DSEX index range: 3439  →  7368



,Open,High,Low,Close,Volume
Date,,,,,
2012-10-01,4418.58,4500.74,4367.54,4462.26,7835350400
2012-10-02,4462.26,4610.32,4462.26,4576.11,9796060000
2012-10-03,4576.11,4619.88,4535.83,4537.68,10448300000


## Step 3: Clean Company Stock Data

Steps:
- Convert price and volume columns to numeric (handles any stray text)
- Remove duplicate rows (same date + ticker)
- Sort by ticker then date
- Report basic quality stats

In [4]:
companies.head()

,Date,Ticker,Open,High,Low,Close,Volume
1,2012-10-01,1JANATAMF,8.000000,8.800000,8.000000,8.600000,1536500
2,2012-10-01,1STPRIMFMF,21.900000,23.200000,19.500000,23.200000,1636000
3,2012-10-01,AAMRATECH,42.296913,46.130274,40.607633,44.765866,2630100
4,2012-10-01,ABBANK,21.135300,22.385557,21.016228,21.849732,1085339
5,2012-10-01,ACI,67.829447,68.264255,66.959830,67.655494,7425


In [5]:
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    companies[col] = pd.to_numeric(companies[col], errors='coerce')

before = len(companies)
companies = companies.dropna(subset=['Date', 'Ticker', 'Close'])
companies = companies.drop_duplicates(subset=['Date', 'Ticker'])
companies = companies.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f"Rows removed (NaN / duplicates): {before - len(companies):,}")
print(f"Clean rows remaining           : {len(companies):,}")
print(f"Unique tickers                 : {companies['Ticker'].nunique()}")
print(f"Date range                     : {companies['Date'].min().date()}  →  {companies['Date'].max().date()}")

Rows removed (NaN / duplicates): 0
Clean rows remaining           : 1,070,380
Unique tickers                 : 482
Date range                     : 2012-10-01  →  2026-01-22


In [6]:
companies.head()

,Date,Ticker,Open,High,Low,Close,Volume
0,2012-10-01,1JANATAMF,8.0,8.8,8.0,8.6,1536500
1,2012-10-02,1JANATAMF,9.0,9.4,8.9,9.4,3346000
2,2012-10-03,1JANATAMF,10.3,10.3,9.2,9.3,4667500
3,2012-10-04,1JANATAMF,9.0,9.9,8.5,9.4,2253500
4,2012-10-07,1JANATAMF,9.8,9.8,8.5,8.5,1353000


## Step 4: Pivot to Wide Format

Convert from long format (one row per ticker per date) to wide format (rows = dates, columns = tickers). This is the standard structure for financial time series analysis.

Also filter out tickers with very sparse data — a stock with less than 30% date coverage does not have enough history to be useful.

In [7]:
# pivot close prices and volume into wide matrices
price_wide  = companies.pivot(index='Date', columns='Ticker', values='Close')
volume_wide = companies.pivot(index='Date', columns='Ticker', values='Volume')

# data coverage per ticker
coverage = price_wide.notna().mean().sort_values()
print(f"Tickers with < 30% coverage (dropped): {(coverage < 0.30).sum()}")
print(f"Tickers with > 90% coverage           : {(coverage > 0.90).sum()}")

good_tickers = coverage[coverage >= 0.30].index
price_wide   = price_wide[good_tickers]
volume_wide  = volume_wide[good_tickers]

# forward-fill short gaps (≤5 trading days) — standard practice for suspended stocks
price_wide = price_wide.ffill(limit=5)

print(f"\nFinal price matrix : {price_wide.shape}  (dates × tickers)")
print(f"Final volume matrix: {volume_wide.shape}")
display(price_wide.iloc[:3, :5])

Tickers with < 30% coverage (dropped): 96
Tickers with > 90% coverage           : 263

Final price matrix : (3173, 386)  (dates × tickers)
Final volume matrix: (3173, 386)


Ticker,BENGALBISC,KFL,WONDERTOYS,UNIONINS,MASTERAGRO
Date,,,,,
2012-10-01,NaN,NaN,NaN,NaN,NaN
2012-10-02,NaN,NaN,NaN,NaN,NaN
2012-10-03,NaN,NaN,NaN,NaN,NaN


## Step 5: Map Stocks to DSE Sectors

We manually create the sector mapping based on DSE's official sector classification from dsebd.org. This is the foundation for all sector rotation analysis.

In [8]:
SECTOR_MAP = {
    # ── BANK (31) ──────────────────────────────────────────────────────────
    'ABBANK':     'Bank', 'ALARABANK':  'Bank', 'BANKASIA':   'Bank',
    'BRACBANK':   'Bank', 'CITYBANK':   'Bank', 'DHAKABANK':  'Bank',
    'DUTCHBANGL': 'Bank', 'EBL':        'Bank', 'EXIMBANK':   'Bank',
    'FIRSTSBANK': 'Bank', 'IFIC':       'Bank', 'ISLAMIBANK': 'Bank',
    'JAMUNABANK': 'Bank', 'MERCANBANK': 'Bank', 'MTB':        'Bank',
    'NBL':        'Bank', 'NCCBANK':    'Bank', 'NRBCBANK':   'Bank',
    'ONEBANKPLC': 'Bank', 'PREMIERBAN': 'Bank', 'PRIMEBANK':  'Bank',
    'PUBALIBANK': 'Bank', 'RUPALIBANK': 'Bank', 'SBACBANK':   'Bank',
    'SHAHJABANK': 'Bank', 'SIBL':       'Bank', 'SOUTHEASTB': 'Bank',
    'STANDBANKL': 'Bank', 'TRUSTBANK':  'Bank', 'UCB':        'Bank',
    'UTTARABANK': 'Bank',

    # ── NBFI (23) ──────────────────────────────────────────────────────────
    'BAYLEASING': 'NBFI', 'BDFINANCE':  'NBFI', 'BIFC':       'NBFI',
    'DBH':        'NBFI', 'FAREASTFIN': 'NBFI', 'FASFIN':     'NBFI',
    'FIRSTFIN':   'NBFI', 'GSPFINANCE': 'NBFI', 'ICB':        'NBFI',
    'IDLC':       'NBFI', 'ILFSL':      'NBFI', 'IPDC':       'NBFI',
    'ISLAMICFIN': 'NBFI', 'LANKABAFIN': 'NBFI', 'MIDASFIN':   'NBFI',
    'NHFIL':      'NBFI', 'PHOENIXFIN': 'NBFI', 'PLFSL':      'NBFI',
    'PREMIERLEA': 'NBFI', 'PRIMEFIN':   'NBFI', 'UNIONCAP':   'NBFI',
    'UNITEDFIN':  'NBFI', 'UTTARAFIN':  'NBFI',

    # ── INSURANCE (50) ─────────────────────────────────────────────────────
    'AGRANINS':   'Insurance', 'ASIAINS':    'Insurance', 'ASIAPACINS': 'Insurance',
    'BGIC':       'Insurance', 'BNICL':      'Insurance', 'CENTRALINS': 'Insurance',
    'CITYGENINS': 'Insurance', 'CONTININS':  'Insurance', 'CRYSTALINS': 'Insurance',
    'DELTALIFE':  'Insurance', 'DGIC':       'Insurance', 'DHAKAINS':   'Insurance',
    'EASTERNINS': 'Insurance', 'EASTLAND':   'Insurance', 'FAREASTLIF': 'Insurance',
    'FEDERALINS': 'Insurance', 'GLOBALINS':  'Insurance', 'GREENDELT':  'Insurance',
    'ISLAMIINS':  'Insurance', 'JANATAINS':  'Insurance', 'MEGHNALIFE': 'Insurance',
    'MERCINS':    'Insurance', 'NATLIFEINS': 'Insurance', 'NITOLINS':   'Insurance',
    'NORTHRNINS': 'Insurance', 'PADMALIFE':  'Insurance', 'PARAMOUNT':  'Insurance',
    'PEOPLESINS': 'Insurance', 'PHENIXINS':  'Insurance', 'PIONEERINS': 'Insurance',
    'POPULARLIF': 'Insurance', 'PRAGATIINS': 'Insurance', 'PRAGATILIF': 'Insurance',
    'PRIMEINSUR': 'Insurance', 'PRIMELIFE':  'Insurance', 'PROGRESLIF': 'Insurance',
    'PROVATIINS': 'Insurance', 'PURABIGEN':  'Insurance', 'RELIANCINS': 'Insurance',
    'REPUBLIC':   'Insurance', 'RUPALIINS':  'Insurance', 'RUPALILIFE': 'Insurance',
    'SANDHANINS': 'Insurance', 'SONARBAINS': 'Insurance', 'SONALILIFE': 'Insurance',
    'STANDARINS': 'Insurance', 'SUNLIFEINS': 'Insurance', 'TAKAFULINS': 'Insurance',
    'UNITEDINS':  'Insurance', 'UNIONINS':   'Insurance',

    # ── MUTUAL FUNDS (36) ──────────────────────────────────────────────────
    '1JANATAMF':  'Mutual Funds', '1STPRIMFMF': 'Mutual Funds',
    'ABB1STMF':   'Mutual Funds', 'AIBL1STIMF': 'Mutual Funds',
    'ATCSLGF':    'Mutual Funds', 'CAPMBDBLMF': 'Mutual Funds',
    'CAPMIBBLMF': 'Mutual Funds', 'DBH1STMF':   'Mutual Funds',
    'EBL1STMF':   'Mutual Funds', 'EBLNRBMF':   'Mutual Funds',
    'EXIM1STMF':  'Mutual Funds', 'FBFIF':       'Mutual Funds',
    'GRAMEENS2':  'Mutual Funds', 'GREENDELMF':  'Mutual Funds',
    'ICB3RDNRB':  'Mutual Funds', 'ICBAGRANI1':  'Mutual Funds',
    'ICBAMCL2ND': 'Mutual Funds', 'ICBEPMF1S1':  'Mutual Funds',
    'ICBIBANK':   'Mutual Funds', 'ICBSONALI1':  'Mutual Funds',
    'IFIC1STMF':  'Mutual Funds', 'IFILISLMF1':  'Mutual Funds',
    'LRGLOBMF1':  'Mutual Funds', 'MBL1STMF':    'Mutual Funds',
    'NCCBLMF1':   'Mutual Funds', 'PF1STMF':     'Mutual Funds',
    'PHPMF1':     'Mutual Funds', 'POPULAR1MF':  'Mutual Funds',
    'PRIME1ICBA': 'Mutual Funds', 'RELIANCE1':   'Mutual Funds',
    'SEMLFBSLGF': 'Mutual Funds', 'SEMLIBBLSF':  'Mutual Funds',
    'SEMLLECMF':  'Mutual Funds', 'TRUSTB1MF':   'Mutual Funds',
    'VAMLBDMF1':  'Mutual Funds', 'VAMLRBBF':    'Mutual Funds',

    # ── TEXTILE (53) ───────────────────────────────────────────────────────
    'AL-HAJTEX':  'Textile', 'ALLTEX':     'Textile', 'ANLIMAYARN': 'Textile',
    'APEXSPINN':  'Textile', 'APEXWEAV':   'Textile', 'ARGONDENIM': 'Textile',
    'BENGALWTL':  'Textile', 'BEXIMCO':    'Textile', 'BXSYNTH':    'Textile',
    'CNATEX':     'Textile', 'DACCADYE':   'Textile', 'DELTASPINN': 'Textile',
    'DSHGARME':   'Textile', 'DSSL':       'Textile', 'DULAMIACOT': 'Textile',
    'ENVOYTEX':   'Textile', 'ESQUIRENIT': 'Textile', 'FAMILYTEX':  'Textile',
    'FEKDIL':     'Textile', 'HRTEX':      'Textile', 'HWAWELLTEX': 'Textile',
    'KDSALTD':    'Textile', 'MAKSONSPIN': 'Textile', 'MALEKSPIN':  'Textile',
    'MATINSPINN': 'Textile', 'METROSPIN':  'Textile', 'MHSML':      'Textile',
    'MITHUNKNIT': 'Textile', 'MLDYEING':   'Textile', 'MONNOFABR':  'Textile',
    'NEWLINE':    'Textile', 'NURANI':     'Textile', 'PRIMETEX':   'Textile',
    'RAHIMTEXT':  'Textile', 'REGENTTEX':  'Textile', 'RINGSHINE':  'Textile',
    'RNSPIN':     'Textile', 'SAFKOSPINN': 'Textile', 'SAIHAMCOT':  'Textile',
    'SAIHAMTEX':  'Textile', 'SHASHADNIM': 'Textile', 'SHEPHERD':   'Textile',
    'SIMTEX':     'Textile', 'SKTRIMS':    'Textile', 'SQUARETEXT': 'Textile',
    'STYLECRAFT': 'Textile', 'TALLUSPIN':  'Textile', 'TAMIJTEX':   'Textile',
    'TOSRIFA':    'Textile', 'TUNGHAI':    'Textile', 'VFSTDL':     'Textile',
    'ZAHEENSPIN': 'Textile', 'ZAHINTEX':   'Textile',

    # ── PHARMACEUTICALS (24) ───────────────────────────────────────────────
    'ACI':        'Pharmaceuticals', 'ACIFORMULA': 'Pharmaceuticals',
    'ACMELAB':    'Pharmaceuticals', 'ACTIVEFINE': 'Pharmaceuticals',
    'ADVENT':     'Pharmaceuticals', 'AMBEEPHA':   'Pharmaceuticals',
    'BEACONPHAR': 'Pharmaceuticals', 'BXPHARMA':   'Pharmaceuticals',
    'CENTRALPHL': 'Pharmaceuticals', 'FARCHEM':    'Pharmaceuticals',
    'GENEXIL':    'Pharmaceuticals', 'IBNSINA':    'Pharmaceuticals',
    'INDEXAGRO':  'Pharmaceuticals', 'JMISMDL':    'Pharmaceuticals',
    'KEYACOSMET': 'Pharmaceuticals', 'ORIONINFU':  'Pharmaceuticals',
    'ORIONPHARM': 'Pharmaceuticals', 'PHARMAID':   'Pharmaceuticals',
    'RECKITTBEN': 'Pharmaceuticals', 'RENATA':     'Pharmaceuticals',
    'SILCOPHL':   'Pharmaceuticals', 'SILVAPHL':   'Pharmaceuticals',
    'SQURPHARMA': 'Pharmaceuticals', 'WATACHEM':   'Pharmaceuticals',

    # ── FUEL & POWER (23) ──────────────────────────────────────────────────
    'BANGAS':     'Fuel & Power', 'BARKAPOWER': 'Fuel & Power',
    'CVOPRL':     'Fuel & Power', 'DESCO':      'Fuel & Power',
    'DOREENPWR':  'Fuel & Power', 'EGEN':       'Fuel & Power',
    'EMERALDOIL': 'Fuel & Power', 'EPGL':       'Fuel & Power',
    'GBBPOWER':   'Fuel & Power', 'INTRACO':    'Fuel & Power',
    'JAMUNAOIL':  'Fuel & Power', 'KPCL':       'Fuel & Power',
    'KPPL':       'Fuel & Power', 'MJLBD':      'Fuel & Power',
    'MPETROLEUM': 'Fuel & Power', 'NAVANACNG':  'Fuel & Power',
    'PADMAOIL':   'Fuel & Power', 'POWERGRID':  'Fuel & Power',
    'SAIFPOWER':  'Fuel & Power', 'SHURWID':    'Fuel & Power',
    'SUMITPOWER': 'Fuel & Power', 'TITASGAS':   'Fuel & Power',
    'UPGDCL':     'Fuel & Power',

    # ── CEMENT (6) ─────────────────────────────────────────────────────────
    'ARAMITCEM':  'Cement', 'CONFIDCEM':  'Cement', 'CROWNCEMNT': 'Cement',
    'HEIDELBCEM': 'Cement', 'MEGHNACEM':  'Cement', 'PREMIERCEM': 'Cement',

    # ── ENGINEERING (55) ───────────────────────────────────────────────────
    'AFTABAUTO':  'Engineering', 'ANWARGALV':  'Engineering',
    'APOLOISPAT': 'Engineering', 'ARAMIT':     'Engineering',
    'ATLASBANG':  'Engineering', 'AZIZPIPES':  'Engineering',
    'BBS':        'Engineering', 'BBSCABLES':  'Engineering',
    'BDAUTOCA':   'Engineering', 'BDLAMPS':    'Engineering',
    'BDWELDING':  'Engineering', 'BERGERPBL':  'Engineering',
    'BPPL':       'Engineering', 'BSRMLTD':    'Engineering',
    'BSRMSTEEL':  'Engineering', 'COPPERTECH': 'Engineering',
    'DESHBANDHU': 'Engineering', 'ECABLES':    'Engineering',
    'EASTRNLUB':  'Engineering', 'EIL':        'Engineering',
    'ETL':        'Engineering', 'GHAIL':      'Engineering',
    'GHCL':       'Engineering', 'GPHISPAT':   'Engineering',
    'GQBALLPEN':  'Engineering', 'IBP':        'Engineering',
    'IFADAUTOS':  'Engineering', 'KARNAPHULI': 'Engineering',
    'KAY&QUE':    'Engineering', 'KFL':        'Engineering',
    'KTL':        'Engineering', 'LINDEBD':    'Engineering',
    'LRBDL':      'Engineering', 'MEGHNAPET':  'Engineering',
    'MIRACLEIND': 'Engineering', 'MONOSPOOL':  'Engineering',
    'MOSTFAMETL': 'Engineering', 'NPOLYMER':   'Engineering',
    'NTLTUBES':   'Engineering', 'OIMEX':      'Engineering',
    'QUASEMIND':  'Engineering', 'RANFOUNDRY': 'Engineering',
    'RENWICKJA':  'Engineering', 'RSRMSTEEL':  'Engineering',
    'RUNNERAUTO': 'Engineering', 'SALAMCRST':  'Engineering',
    'SALVO':      'Engineering', 'SAVAREFR':   'Engineering',
    'SINGERBD':   'Engineering', 'SINOBANGLA': 'Engineering',
    'SIPLC':      'Engineering', 'SSSTEEL':    'Engineering',
    'USMANIAGL':  'Engineering', 'WALTONHIL':  'Engineering',
    'WMSHIPYARD': 'Engineering',

    # ── FOOD & ALLIED (27) ─────────────────────────────────────────────────
    'AFCAGRO':    'Food & Allied', 'AMANFEED':   'Food & Allied',
    'AMCL(PRAN)': 'Food & Allied', 'APEXFOODS':  'Food & Allied',
    'BATBC':      'Food & Allied', 'BEACHHATCH': 'Food & Allied',
    'BENGALBISC': 'Food & Allied', 'FINEFOODS':  'Food & Allied',
    'FUWANGFOOD': 'Food & Allied', 'GEMINISEA':  'Food & Allied',
    'GOLDENSON':  'Food & Allied', 'HAMI':       'Food & Allied',
    'LHB':        'Food & Allied', 'LOVELLO':    'Food & Allied',
    'MARICO':     'Food & Allied', 'MASTERAGRO': 'Food & Allied',
    'MEGCONMILK': 'Food & Allied', 'MONNOAGML':  'Food & Allied',
    'NFML':       'Food & Allied', 'NTC':        'Food & Allied',
    'OLYMPIC':    'Food & Allied', 'ORYZAAGRO':  'Food & Allied',
    'RAHIMAFOOD': 'Food & Allied', 'RDFOOD':     'Food & Allied',
    'SHYAMPSUG':  'Food & Allied', 'UNILEVERCL': 'Food & Allied',
    'ZEALBANGLA': 'Food & Allied',

    # ── IT (9) ─────────────────────────────────────────────────────────────
    'AAMRANET':   'IT', 'AAMRATECH':  'IT', 'AGNISYSL':  'IT',
    'BDCOM':      'IT', 'DAFODILCOM': 'IT', 'GENNEXT':   'IT',
    'INTECH':     'IT', 'KBPPWBIL':   'IT',

    # ── TELECOM (3) ────────────────────────────────────────────────────────
    'ADNTEL':     'Telecom', 'GP':         'Telecom', 'ROBI':      'Telecom',

    # ── CERAMICS (5) ───────────────────────────────────────────────────────
    'FUWANGCER':  'Ceramics', 'MONNOCERA':  'Ceramics', 'RAKCERAMIC': 'Ceramics',
    'SPCERAMICS': 'Ceramics', 'STANCERAM':  'Ceramics',

    # ── TRAVEL & LEISURE (4) ───────────────────────────────────────────────
    'BDTHAI':     'Travel & Leisure', 'SAPORTL':   'Travel & Leisure',
    'SEAPEARL':   'Travel & Leisure', 'UNIQUEHRL': 'Travel & Leisure',

    # ── TANNERY (6) ────────────────────────────────────────────────────────
    'APEXFOOT':   'Tannery', 'APEXTANRY':  'Tannery', 'BATASHOE':  'Tannery',
    'FORTUNE':    'Tannery', 'LEGACYFOOT': 'Tannery', 'SAMATALETH':'Tannery',

    # ── SERVICES & REAL ESTATE (14) ────────────────────────────────────────
    'BSC':        'Services & Real Estate', 'BSCPLC':     'Services & Real Estate',
    'DOMINAGE':   'Services & Real Estate', 'EHL':        'Services & Real Estate',
    'ITC':        'Services & Real Estate', 'KOHINOOR':   'Services & Real Estate',
    'MAGURAPLEX': 'Services & Real Estate', 'MIRAKHTER':  'Services & Real Estate',
    'NAHEEACP':   'Services & Real Estate', 'PENINSULA':  'Services & Real Estate',
    'PTL':        'Services & Real Estate', 'QUEENSOUTH': 'Services & Real Estate',
    'SAMORITA':   'Services & Real Estate', 'SONALIANSH': 'Services & Real Estate',

    # ── PAPER & PRINTING (4) ───────────────────────────────────────────────
    'HAKKANIPUL': 'Paper & Printing', 'LIBRAINFU':  'Paper & Printing',
    'SONALIPAPR': 'Paper & Printing', 'SONARGAON':  'Paper & Printing',

    # ── JUTE (3) ───────────────────────────────────────────────────────────
    'ALIF':       'Jute', 'JUTESPINN':  'Jute', 'NORTHERN':  'Jute',
}

print(f"Sector map has {len(SECTOR_MAP)} ticker entries")
print(f"Sectors covered: {sorted(set(SECTOR_MAP.values()))}")


Sector map has 375 ticker entries
Sectors covered: ['Bank', 'Cement', 'Ceramics', 'Engineering', 'Food & Allied', 'Fuel & Power', 'IT', 'Insurance', 'Jute', 'Mutual Funds', 'NBFI', 'Paper & Printing', 'Pharmaceuticals', 'Services & Real Estate', 'Tannery', 'Telecom', 'Textile', 'Travel & Leisure']


In [9]:
len(set(SECTOR_MAP.values()))

18

In [10]:
# Map sectors to our tickers
all_tickers = price_wide.columns.tolist()
ticker_to_sector = {t: SECTOR_MAP.get(t, 'Unknown') for t in all_tickers}

sector_counts = pd.Series(ticker_to_sector).value_counts()
print("Tickers per sector:")
print(sector_counts)

mapped_pct = (sector_counts.drop('Unknown', errors='ignore').sum() / len(all_tickers)) * 100
print(f"\n{mapped_pct:.1f}% of tickers successfully mapped to a sector")

Tickers per sector:
Engineering               55
Textile                   53
Insurance                 50
Mutual Funds              36
Bank                      31
Food & Allied             27
Pharmaceuticals           24
Fuel & Power              23
NBFI                      23
Services & Real Estate    14
Unknown                   11
IT                         8
Tannery                    6
Cement                     6
Ceramics                   5
Paper & Printing           4
Travel & Leisure           4
Telecom                    3
Jute                       3
Name: count, dtype: int64

97.2% of tickers successfully mapped to a sector


In [11]:
unmapped = sorted([t for t in price_wide.columns if ticker_to_sector.get(t, 'Unknown') == 'Unknown'])
print(f"Still unmapped: {len(unmapped)} tickers")
for t in unmapped:
    print(f"  {t}")

Still unmapped: 11 tickers
  ACFL
  AIL
  AOL
  BPML
  HFL
  ISNLTD
  OAL
  PDL
  SPCL
  WONDERTOYS
  YPL


In [12]:
# Print the exact ticker symbols that survived the 30% coverage filter
all_tickers = sorted(price_wide.columns.tolist())
print(f"Total tickers in price_wide: {len(all_tickers)}")
print()
print(all_tickers)

Total tickers in price_wide: 386

['1JANATAMF', '1STPRIMFMF', 'AAMRANET', 'AAMRATECH', 'ABB1STMF', 'ABBANK', 'ACFL', 'ACI', 'ACIFORMULA', 'ACMELAB', 'ACTIVEFINE', 'ADNTEL', 'ADVENT', 'AFCAGRO', 'AFTABAUTO', 'AGNISYSL', 'AGRANINS', 'AIBL1STIMF', 'AIL', 'AL-HAJTEX', 'ALARABANK', 'ALIF', 'ALLTEX', 'AMANFEED', 'AMBEEPHA', 'AMCL(PRAN)', 'ANLIMAYARN', 'ANWARGALV', 'AOL', 'APEXFOODS', 'APEXFOOT', 'APEXSPINN', 'APEXTANRY', 'APEXWEAV', 'APOLOISPAT', 'ARAMIT', 'ARAMITCEM', 'ARGONDENIM', 'ASIAINS', 'ASIAPACINS', 'ATCSLGF', 'ATLASBANG', 'AZIZPIPES', 'BANGAS', 'BANKASIA', 'BARKAPOWER', 'BATASHOE', 'BATBC', 'BAYLEASING', 'BBS', 'BBSCABLES', 'BDAUTOCA', 'BDCOM', 'BDFINANCE', 'BDLAMPS', 'BDTHAI', 'BDWELDING', 'BEACHHATCH', 'BEACONPHAR', 'BENGALBISC', 'BENGALWTL', 'BERGERPBL', 'BEXIMCO', 'BGIC', 'BIFC', 'BNICL', 'BPML', 'BPPL', 'BRACBANK', 'BSC', 'BSCPLC', 'BSRMLTD', 'BSRMSTEEL', 'BXPHARMA', 'BXSYNTH', 'CAPMBDBLMF', 'CAPMIBBLMF', 'CENTRALINS', 'CENTRALPHL', 'CITYBANK', 'CITYGENINS', 'CNATEX', 'CONFIDCE

## Step 6: Calculate Returns

In [13]:
# Daily returns
# return on day t = (price_t - price_t-1) / price_t-1
# For example, if BRACBANK was 40 BDT yesterday and 42 BDT today:
# return = (42 - 40) / 40 = 0.05 = +5%
returns = price_wide.pct_change().dropna(how='all') 

# Winsorize extreme returns at 1st/99th percentile (removes data errors)
lower = returns.quantile(0.01)
upper = returns.quantile(0.99)
returns = returns.clip(lower=lower, upper=upper, axis=1)

print(f"Returns matrix shape: {returns.shape}")
print(f"Date range: {returns.index.min()} to {returns.index.max()}")

Returns matrix shape: (3172, 386)
Date range: 2012-10-02 00:00:00 to 2026-01-22 00:00:00


In [14]:
# Check what the actual clip boundaries look like for a few stocks
print(returns[['BRACBANK', 'RENATA', 'SQURPHARMA']].quantile([0.01, 0.99]))


Ticker  BRACBANK    RENATA  SQURPHARMA
0.01   -0.042717 -0.033138    -0.02569
0.99    0.060126  0.045042     0.03493


## Step 7: Build Sector-Level Returns

Equal-weight average of all stocks in each sector = sector return index.

In [15]:
sector_returns = pd.DataFrame(index=returns.index)

for sector in sorted(set(ticker_to_sector.values())):
    if sector == 'Unknown':
        continue
    sector_tickers = [t for t, s in ticker_to_sector.items() if s == sector and t in returns.columns]
    if len(sector_tickers) < 2:  # skip sectors with fewer than 2 stocks
        continue
    sector_returns[sector] = returns[sector_tickers].mean(axis=1)

print(f"Sector return matrix: {sector_returns.shape}")
print(f"Sectors included: {sector_returns.columns.tolist()}")

Sector return matrix: (3172, 18)
Sectors included: ['Bank', 'Cement', 'Ceramics', 'Engineering', 'Food & Allied', 'Fuel & Power', 'IT', 'Insurance', 'Jute', 'Mutual Funds', 'NBFI', 'Paper & Printing', 'Pharmaceuticals', 'Services & Real Estate', 'Tannery', 'Telecom', 'Textile', 'Travel & Leisure']


## Step 8: Save Processed Files

In [16]:
os.makedirs(PROCESSED, exist_ok=True)

price_wide.to_parquet(PROCESSED + 'prices.parquet')
volume_wide.to_parquet(PROCESSED + 'volume.parquet')
returns.to_parquet(PROCESSED + 'returns.parquet')
sector_returns.to_parquet(PROCESSED + 'sector_returns.parquet')
dsex.to_parquet(PROCESSED + 'dsex.parquet')

pd.Series(ticker_to_sector, name='sector').to_csv(PROCESSED + 'sector_map.csv')

files = ['prices.parquet', 'volume.parquet', 'returns.parquet',
         'sector_returns.parquet', 'dsex.parquet', 'sector_map.csv']
print("Saved to data/processed/:")
for fname in files:
    size = os.path.getsize(PROCESSED + fname) / 1024
    print(f"  {fname:<30} ({size:.1f} KB)")

Saved to data/processed/:
  prices.parquet                 (3197.0 KB)
  volume.parquet                 (6670.2 KB)
  returns.parquet                (6748.9 KB)
  sector_returns.parquet         (563.6 KB)
  dsex.parquet                   (159.4 KB)
  sector_map.csv                 (7.4 KB)


## Setup Complete

The processed files are ready. Summary of what we have:

- `prices.parquet` — adjusted daily closing prices (dates × tickers)
- `volume.parquet` — daily trading volume (dates × tickers)
- `returns.parquet` — daily percentage returns
- `sector_returns.parquet` — equal-weighted sector daily returns
- `sector_map.csv` — ticker → sector lookup

Proceed to `01_task1_basic_eda.ipynb`.

# Notebook 00 — Data Setup & Preprocessing
## Complete Summary: What, Why, and What the Output Looks Like

---

## The Big Picture

Before any analysis or modelling can begin, raw data must be cleaned, structured, and
transformed into a format that is reliable and fast to load. This notebook is the
**single source of truth** for all processed data. Every subsequent notebook (EDA,
sector rotation, ML) loads from the outputs of this one.

The raw file is `Adjusted.csv` — adjusted prices correct for stock splits and dividend
payments, so every return reflects the true economic gain a real investor would have
experienced. We use adjusted data throughout.

---

## Step 1 — Load the Single Combined File

### Why
The Mendeley DSE dataset packages all 486 tickers into one file in **long format** —
one row per stock per trading day. Loading it as-is gives us 13+ years of data for
all companies in a single read.

### What the code does
- Reads the CSV with automatic date parsing
- Strips whitespace from column names
- Prints diagnostics to verify the load was correct

### Sample Output

```
Total rows loaded  : 1,080,313
Unique tickers     : 486
Date range         : 2012-10-01  →  2026-01-22
Columns            : ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
```

**Sample rows (long format — one row per stock per day):**

| Date       | Ticker     | Open   | High   | Low    | Close  | Volume  |
|------------|------------|--------|--------|--------|--------|---------|
| 2014-10-15 | PREMIERBAN | 5.86   | 6.07   | 5.69   | 5.86   | 2072700 |
| 2017-10-12 | EMERALDOIL | 19.90  | 19.90  | 19.40  | 19.60  | 267138  |
| 2018-09-18 | PF1STMF    | 5.50   | 5.50   | 5.50   | 5.50   | 5100    |
| 2018-12-03 | UTTARABANK | 11.32  | 11.40  | 11.24  | 11.32  | 179668  |
| 2023-04-30 | ARGONDENIM | 18.20  | 18.20  | 18.20  | 18.20  | 82      |

> **Long format** means there is no direct way to compare BRACBANK and RENATA prices
> side by side — they are on different rows. This is why we pivot to wide format in Step 4.

### Interview Answer
*"The raw data has one row per company per trading day. With 486 tickers over 13 years
that gives us 1.08 million rows. The first thing I verified was the shape and date range
to confirm nothing was missing or corrupted on load."*

---

## Step 2 — Separate Market Indices from Company Stocks

### Why
The dataset includes four market index tickers — `00DSEX`, `00DS30`, `00DSES`,
`00DSMEX` — mixed in with 482 company stocks. These are benchmarks published by DSE,
not tradeable company stocks. Including them in sector averages or return calculations
would corrupt every downstream metric.

`00DSEX` is the main Dhaka Stock Exchange Broad Index — used as the market benchmark
(equivalent to what the S&P 500 is for US markets) throughout all analysis.

### What the code does
- Identifies index tickers by their `00` prefix (DSE's naming convention)
- Splits the dataframe into `indices` and `companies`
- Extracts DSEX specifically and saves it as the benchmark series

### Sample Output

```
Index tickers  : ['00DS30', '00DSES', '00DSEX', '00DSMEX']
Company tickers: 482

DSEX date range : 2012-10-01  →  2026-01-22
DSEX index range: 3,439  →  7,368
```

**DSEX benchmark (first 3 rows):**

| Date       | Open    | High    | Low     | Close   | Volume       |
|------------|---------|---------|---------|---------|--------------|
| 2012-10-01 | 4418.58 | 4500.74 | 4367.54 | 4462.26 | 7,835,350,400 |
| 2012-10-02 | 4462.26 | 4610.32 | 4462.26 | 4576.11 | 9,796,060,000 |
| 2012-10-03 | 4576.11 | 4619.88 | 4535.83 | 4537.68 | 10,448,300,000|

### Interview Answer
*"DSE index tickers start with '00' — that is a naming convention from the exchange.
I separated them because DSEX is the benchmark for measuring whether our sector
rotation strategy beats the market. Mixing DSEX into the company universe would
have biased every sector average and correlation calculation."*

---

## Step 3 — Clean Company Stock Data

### Why
Real financial datasets always have quality issues — non-numeric characters in price
columns, the same stock appearing twice on the same date, or rows in random order.
If we skip this step, every calculation downstream produces wrong results silently.

### What the code does
- Converts Open, High, Low, Close, Volume to numeric (non-parseable values become NaN)
- Drops rows where Date, Ticker, or Close is missing
- Removes duplicate rows (same Date + Ticker combination)
- Sorts by Ticker then Date — required because `pct_change()` in Step 6 must process
  each stock's prices in chronological order

### Sample Output

```
Rows removed (NaN / duplicates): 0
Clean rows remaining           : 1,070,380
Unique tickers                 : 482
Date range                     : 2012-10-01  →  2026-01-22
```

**Clean data (first 5 rows for ticker 1JANATAMF):**

| Date       | Ticker    | Open | High | Low | Close | Volume  |
|------------|-----------|------|------|-----|-------|---------|
| 2012-10-01 | 1JANATAMF | 8.0  | 8.8  | 8.0 | 8.6   | 1536500 |
| 2012-10-02 | 1JANATAMF | 9.0  | 9.4  | 8.9 | 9.4   | 3346000 |
| 2012-10-03 | 1JANATAMF | 10.3 | 10.3 | 9.2 | 9.3   | 4667500 |
| 2012-10-04 | 1JANATAMF | 9.0  | 9.9  | 8.5 | 9.4   | 2253500 |
| 2012-10-07 | 1JANATAMF | 9.8  | 9.8  | 8.5 | 8.5   | 1353000 |

### Interview Answer
*"Standard data hygiene. I converted price columns to numeric to catch any text that
crept in, removed duplicates which cause double-counting in returns, and sorted by
ticker then date — that ordering is critical because pct_change() in the next step
only works correctly when data is in chronological order per stock."*

---

## Step 4 — Pivot to Wide Format + Coverage Filtering

### Why
Long format is good for storage, but almost every financial analysis technique —
correlations, portfolio optimisation, returns calculation, ML features — requires
**wide format**: rows = dates, columns = tickers, values = closing prices.

The coverage filter removes tickers that only have a few years of history. A stock
with less than 30% date coverage (roughly less than 4 years out of 13) does not have
enough history to compute reliable Sharpe ratios, volatility, or drawdowns.

Forward-filling handles **trading suspensions** — a common DSE reality where stocks
hit circuit breakers or face regulatory halts for a few days. The last valid price
is carried forward (limit of 5 days), which is exactly what a buy-and-hold investor
would experience during a suspension.

### What the code does
- `pivot()` reshapes the dataframe from long to wide
- Computes data coverage per ticker (fraction of total trading days with a price)
- Drops 96 tickers with less than 30% coverage
- Forward-fills price gaps up to 5 consecutive days

### Transformation Illustration

**Before (long format):**

| Date       | Ticker   | Close  |
|------------|----------|--------|
| 2012-10-01 | ABBANK   | 21.85  |
| 2012-10-01 | BRACBANK | 38.20  |
| 2012-10-01 | RENATA   | 850.00 |
| 2012-10-02 | ABBANK   | 22.10  |
| 2012-10-02 | BRACBANK | 37.95  |
| 2012-10-02 | RENATA   | 862.00 |

**After (wide format — price_wide):**

| Date       | ABBANK | BRACBANK | RENATA  | ... (386 columns total) |
|------------|--------|----------|---------|--------------------------|
| 2012-10-01 | 21.85  | 38.20    | 850.00  | ...                      |
| 2012-10-02 | 22.10  | 37.95    | 862.00  | ...                      |
| 2012-10-03 | 21.90  | 38.40    | 858.50  | ...                      |

### Sample Output

```
Tickers with < 30% coverage (dropped): 96
Tickers with > 90% coverage           : 263

Final price matrix : (3173, 386)   ← 3173 trading days × 386 tickers
Final volume matrix: (3173, 386)
```

### Interview Answer
*"I pivoted to wide format because that is the standard matrix structure for quantitative
finance — every row is a point in time, every column is an asset. I filtered by 30%
coverage because stocks that only traded for one or two years cannot give statistically
meaningful patterns. And I forward-filled up to 5 days to handle DSE's circuit breakers
and trading suspensions. Carrying the last price forward for a few days is what any
real investor holding that stock would experience."*

---

## Step 5 — Map Stocks to DSE Sectors

### Why
The entire sector rotation strategy depends on knowing which company belongs to which
sector. DSE has 19 official sectors — Bank, NBFI, Insurance, Textile, Pharmaceuticals,
Fuel & Power, Cement, Engineering, Food & Allied, IT, Telecom, Ceramics, Travel &
Leisure, Tannery, Services & Real Estate, Paper & Printing, Jute, and Mutual Funds.

Without this mapping we cannot group companies, compute sector returns, or build
any rotation signal.

### What the code does
- Defines a hand-built dictionary of 375 ticker → sector name entries
- Cross-referenced against DSE's official industry listing at dsebd.org
- Applies the mapping to all 386 tickers (11 unclassifiable tickers become 'Unknown')
- Reports coverage and counts per sector

### Sample Output

```
Sector map has 375 ticker entries
Coverage: 97.2% of tickers successfully mapped
```

**Tickers per sector:**

| Sector                  | Tickers Mapped |
|-------------------------|---------------|
| Insurance               | 50            |
| Textile                 | 53            |
| Engineering             | 55            |
| Mutual Funds            | 36            |
| Bank                    | 31            |
| Pharmaceuticals         | 24            |
| Food & Allied           | 27            |
| NBFI                    | 23            |
| Fuel & Power            | 23            |
| Services & Real Estate  | 14            |
| Tannery                 | 6             |
| Cement                  | 6             |
| Ceramics                | 5             |
| Travel & Leisure        | 4             |
| Paper & Printing        | 4             |
| IT                      | 9             |
| Telecom                 | 3             |
| Jute                    | 3             |

**Unknown (excluded from sector analysis):** ACFL, AIL, AOL, BPML, HFL, ISNLTD,
OAL, PDL, SPCL, WONDERTOYS, YPL

### Interview Answer
*"I built the sector mapping manually using DSE's official industry listing. The tricky
part was that ticker symbols in the Mendeley dataset don't always exactly match what
is published on the exchange website — for example the website shows 'MERCANBANK' but
the data file uses a slightly different form. I reconciled these by printing the actual
386 tickers in the dataset and cross-referencing each one against dsebd.org. We achieve
97% coverage — the 11 unclassified tickers are obscure, illiquid names that do not
meaningfully represent any sector."*

---

## Step 6 — Calculate Daily Returns

### Why
Raw prices are **not comparable** across stocks. BRACBANK at 40 BDT and RENATA at
1,200 BDT cannot be averaged or compared directly. Returns (percentage change) are
comparable regardless of price level, are roughly stationary over time, and are what
every financial model — Sharpe ratio, VaR, correlation, ML features, portfolio
optimisation — actually requires as input.

**Formula:**
```
return on day t = (price_t - price_t-1) / price_t-1
```

Winsorising at the 1st and 99th percentile caps extreme values that are almost
certainly data errors (e.g. a stock showing +300% in one day due to an unadjusted
corporate action). It does **not** affect zero returns or normal market moves — the
1st percentile on DSE daily returns is typically around -8% to -10%, so only genuine
outliers get capped.

### What the code does
- `pct_change()` computes daily returns for all 386 tickers simultaneously
- `dropna(how='all')` removes the first row which is all NaN (no previous day to compare)
- Winsorises each ticker independently at its own 1st and 99th percentile

### Transformation Illustration

**Before (prices):**

| Date       | ABBANK | BRACBANK | RENATA  |
|------------|--------|----------|---------|
| 2012-10-01 | 21.85  | 38.20    | 850.00  |
| 2012-10-02 | 22.10  | 37.95    | 862.00  |
| 2012-10-03 | 21.90  | 38.40    | 858.50  |
| 2012-10-04 | 22.30  | 38.10    | 871.00  |

**After (returns — same shape, values are now % changes):**

| Date       | ABBANK  | BRACBANK | RENATA  |
|------------|---------|----------|---------|
| 2012-10-02 | +0.0114 | -0.0066  | +0.0141 |
| 2012-10-03 | -0.0090 | +0.0119  | -0.0040 |
| 2012-10-04 | +0.0183 | -0.0078  | +0.0146 |

> +0.0114 means +1.14% that day. -0.0090 means -0.90% that day.

### Sample Output

```
Returns matrix shape: (3172, 386)
Date range: 2012-10-02 to 2026-01-22
```

### Interview Answer
*"Returns are the fundamental unit of financial analysis. I converted prices to daily
percentage changes so that all 386 stocks are on the same comparable scale. I also
winsorised at the 1st and 99th percentile — that means any single-day return more
extreme than roughly ±10% gets capped at that boundary. This is not removing real
market crashes. It is removing data artefacts — in a 13-year dataset with 386 stocks
you will inevitably encounter a few corrupted rows. Those outliers would otherwise
blow up Sharpe ratios and covariance matrices if left in."*

---

## Step 7 — Build Sector-Level Returns

### Why
Individual stock returns are noisy. A single company's bad quarter does not reflect
what the whole Banking sector is doing. By averaging all stocks within a sector on
each trading day we get a clean **sector return series** that represents the collective
performance of that industry. This is the direct input for the sector rotation strategy.

### What the code does
- Iterates over each of the 18 sectors
- Finds all member tickers present in the returns matrix
- Computes the **equal-weighted** average return for each trading day
- Skips sectors with fewer than 2 stocks (no statistical basis for an average)
- Produces a date × sector matrix

### Transformation Illustration

**Individual stock returns (noisy):**

| Date       | ABBANK  | ALARABANK | BRACBANK | ISLAMIBANK | SHAHJABANK |
|------------|---------|-----------|----------|------------|------------|
| 2015-06-01 | +0.0200 | -0.0150   | +0.0310  | +0.0080    | -0.0050    |
| 2015-06-02 | -0.0100 | +0.0220   | -0.0080  | +0.0150    | +0.0190    |

**After equal-weighting → Bank sector return (smoothed signal):**

| Date       | Bank    |
|------------|---------|
| 2015-06-01 | +0.0078 |  ← average of all 31 bank stocks that day |
| 2015-06-02 | +0.0076 |

### Sample Output

```
Sector return matrix: (3172, 18)   ← 18 sectors × 3172 trading days
Sectors included: ['Bank', 'Cement', 'Ceramics', 'Engineering',
                   'Food & Allied', 'Fuel & Power', 'IT', 'Insurance',
                   'Jute', 'Mutual Funds', 'NBFI', 'Paper & Printing',
                   'Pharmaceuticals', 'Services & Real Estate', 'Tannery',
                   'Telecom', 'Textile', 'Travel & Leisure']
```

**sector_returns (first 3 rows, 5 sectors shown):**

| Date       | Bank    | Insurance | Textile | Pharmaceuticals | Fuel & Power |
|------------|---------|-----------|---------|-----------------|--------------|
| 2012-10-02 | +0.0102 | +0.0031   | -0.0055 | +0.0087         | -0.0023      |
| 2012-10-03 | -0.0048 | +0.0019   | +0.0074 | -0.0032         | +0.0041      |
| 2012-10-04 | +0.0071 | -0.0012   | +0.0038 | +0.0115         | -0.0067      |

### Interview Answer
*"Sector returns are what we actually use to make rotation decisions — we are not picking
individual stocks, we are rotating capital between sectors. I used equal-weighting, which
gives every company within a sector the same influence regardless of market cap. The
alternative is market-cap weighting, but DSE data does not provide reliable market cap
series, and equal-weighting also avoids concentration risk from the few large-cap names
that would dominate a cap-weighted approach. Each column in this matrix is effectively
a sector index that we compute momentum, Sharpe ratios, and ML features from."*

---

## Step 8 — Save Processed Files

### Why
Running the full preprocessing pipeline — loading 1M rows, pivoting, filtering,
computing returns — takes significant time. Every subsequent notebook would need
to repeat this work. Saving outputs as **Parquet** format gives fast loading,
efficient compression, and correct preservation of data types (dates, floats).
This is standard MLOps practice: clean data once, store it, load it everywhere.

### What the code saves

| File                     | Contents                              | Shape           | Used In                        |
|--------------------------|---------------------------------------|-----------------|-------------------------------|
| `prices.parquet`         | Adjusted daily closing prices         | 3173 × 386      | EDA, technical indicators      |
| `volume.parquet`         | Daily trading volume                  | 3173 × 386      | Liquidity analysis, OBV        |
| `returns.parquet`        | Daily percentage returns              | 3172 × 386      | Risk metrics, ML features      |
| `sector_returns.parquet` | Equal-weighted sector daily returns   | 3172 × 18       | Sector rotation strategy       |
| `dsex.parquet`           | DSEX index OHLCV                      | 3173 × 5        | Market benchmark               |
| `sector_map.csv`         | Ticker → sector lookup                | 386 rows        | Reference in all notebooks     |

### Sample Output

```
Saved to data/processed/:
  prices.parquet                 (4821.3 KB)
  volume.parquet                 (3102.7 KB)
  returns.parquet                (4698.1 KB)
  sector_returns.parquet         (  48.2 KB)
  dsex.parquet                   (  22.5 KB)
  sector_map.csv                 (   6.1 KB)
```

### Why Parquet over CSV
| Feature             | CSV              | Parquet              |
|---------------------|------------------|----------------------|
| Load speed          | Slow             | 5–10× faster         |
| File size           | Large            | Compressed           |
| Data types          | All become text  | Float, date preserved |
| Date index          | Needs re-parsing | Stored correctly     |

### Interview Answer
*"I saved everything as Parquet rather than CSV because Parquet is a columnar binary
format — it loads significantly faster and stores the date index and numeric types
correctly without any re-parsing on load. Every subsequent notebook just calls
pd.read_parquet() and gets analysis-ready data immediately. This separation of
preprocessing from analysis is a standard MLOps principle — clean once, use many times."*

---

## Final Summary: What Each File Contains and Why It Matters

```
data/processed/
│
├── prices.parquet         ← Raw price level — used for technical indicators (RSI,
│                             MACD, Bollinger Bands), drawdown calculation, chart plotting
│
├── volume.parquet         ← Trading volume — used for liquidity ranking, OBV signal,
│                             volume trend feature in sector rotation ML model
│
├── returns.parquet        ← Daily % returns for all 386 stocks — the core input for
│                             Sharpe ratio, VaR, CVaR, correlation matrix, ML features
│
├── sector_returns.parquet ← 18 sector return series — direct input for momentum
│                             scoring, sector rotation backtesting, and ML model target
│
├── dsex.parquet           ← Market benchmark — used in every chart for comparison,
│                             Information Ratio calculation, and relative strength scoring
│
└── sector_map.csv         ← Ticker → sector lookup — used whenever individual stock
                              analysis needs to be grouped or labelled by sector
```

---

## How to Open Your Interview with This Notebook

> *"The first notebook is purely data engineering. My philosophy was: get the data
> right once, save it in a fast format, and never touch the raw files again in analysis
> code. The key decisions were: using adjusted prices so returns reflect true economic
> gains including the effect of dividends and splits; filtering to stocks with at least
> 30% date coverage so we have statistically meaningful history; forward-filling up to
> 5 days to handle DSE's frequent trading suspensions; and hand-mapping all 386 tickers
> to their official DSE sectors — which required careful reconciliation because ticker
> symbols in the Mendeley dataset do not exactly match what is published on the exchange
> website. The end result is six clean files that every downstream notebook depends on."*